# Etapa 2 — Aprendizado Linear

- **Alvo contínuo:** `y = log(1 + homicídios dolosos em 2023)` no município.
- **Atributos:** `log(população)` e as contagens de 2020, 2021 e 2022 (homicídios, tentativas, mortes a esclarecer, latrocínios e mandados) em `log(1+x)`.
- **Classificação (logística):** o alvo binário da Etapa 1 (presença de facção).

Divisão fixa treino/teste 80/20 (`random_state=42`).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from utils import montar_tabela, atributos_numericos_log, ATRIBUTOS_CAT, DESCRICAO, SEED
np.random.seed(SEED)
pd.set_option("display.width", 140)
d = montar_tabela()
print(d.shape)

(808, 115)


In [2]:
from sklearn.model_selection import train_test_split, KFold
X_all = atributos_numericos_log(d)
y_all = np.log1p(d["homicidio_2023"].values)
X_tr, X_te, y_tr, y_te, yb_tr, yb_te = train_test_split(
    X_all.values, y_all, d["ALVO"].values, test_size=0.2, random_state=SEED, stratify=d["ALVO"].values)
cols = list(X_all.columns)
print(len(cols), "atributos")

def r2(y, yhat):
    return 1 - ((y - yhat) ** 2).sum() / ((y - y.mean()) ** 2).sum()

16 atributos


## 1. Regressão linear univariada (fórmula fechada)

$$w_1 = \frac{\sum (x_i - \bar x)(y_i - \bar y)}{\sum (x_i - \bar x)^2}, \qquad w_0 = \bar y - w_1 \bar x$$

In [3]:
j = cols.index("log_homicidio_2022")
x_tr, x_te = X_tr[:, j], X_te[:, j]
w1 = ((x_tr - x_tr.mean()) * (y_tr - y_tr.mean())).sum() / ((x_tr - x_tr.mean()) ** 2).sum()
w0 = y_tr.mean() - w1 * x_tr.mean()
print(f"log(1+hom_2023) = {w0:.4f} + {w1:.4f} * log(1 + hom_2022)")
r2_uni = r2(y_te, w0 + w1 * x_te)
print(f"R² no teste: {r2_uni:.3f}")
plt.figure(figsize=(5.5, 3.5))
plt.scatter(x_te, y_te, s=8, alpha=.4)
xx = np.linspace(x_te.min(), x_te.max(), 50); plt.plot(xx, w0 + w1 * xx, "r")
plt.xlabel("log(1 + homicídios 2022)"); plt.ylabel("log(1 + homicídios 2023)")
plt.tight_layout(); plt.savefig("figuras/etapa2_univariada.png", dpi=150)

log(1+hom_2023) = 0.2818 + 0.8123 * log(1 + hom_2022)
R² no teste: 0.672


## 2. Regressão multivariada com gradiente descendente (do zero)

In [4]:
mu, sd = X_tr.mean(0), X_tr.std(0)
Z_tr, Z_te = (X_tr - mu) / sd, (X_te - mu) / sd
A_tr = np.c_[np.ones(len(Z_tr)), Z_tr]; A_te = np.c_[np.ones(len(Z_te)), Z_te]

def gd_linear(A, y, alpha=0.05, n_iter=5000, lam=0.0):
    w = np.zeros(A.shape[1]); hist = []
    N = len(y)
    for _ in range(n_iter):
        erro = y - A @ w
        w = w + alpha * (A.T @ erro / N - lam * np.r_[0, w[1:]])
        hist.append((erro ** 2).mean())
    return w, hist

w_gd, hist = gd_linear(A_tr, y_tr)
r2_multi = r2(y_te, A_te @ w_gd)
print(f"R² no teste (GD): {r2_multi:.3f}")
plt.figure(figsize=(5, 3)); plt.plot(hist); plt.yscale("log")
plt.xlabel("iteração"); plt.ylabel("erro quadrático médio"); plt.title("Convergência do GD")
plt.tight_layout(); plt.savefig("figuras/etapa2_convergencia_gd.png", dpi=150)

R² no teste (GD): 0.763


## 3. Regularização L2 (Ridge) e curva de coeficientes × λ

Com 16 atributos muito correlacionados (o mesmo evento em três anos seguidos), a regularização tem efeito visível. Usamos a solução fechada $w = (A^TA/N + \lambda I')^{-1} A^Ty/N$ (o intercepto não é penalizado) e conferimos com o GD.

In [5]:
def ridge_fechado(A, y, lam):
    N, p = A.shape
    I = np.eye(p); I[0, 0] = 0
    return np.linalg.solve(A.T @ A / N + lam * I, A.T @ y / N)

lams = np.logspace(-4, 2, 40)
kf = KFold(5, shuffle=True, random_state=SEED)
coefs = np.array([ridge_fechado(A_tr, y_tr, l)[1:] for l in lams])
r2_cv = [np.mean([r2(y_tr[b], A_tr[b] @ ridge_fechado(A_tr[a], y_tr[a], l)) for a, b in kf.split(A_tr)]) for l in lams]
lam_otimo = lams[int(np.argmax(r2_cv))]
w_ridge = ridge_fechado(A_tr, y_tr, lam_otimo)
r2_ridge = r2(y_te, A_te @ w_ridge)
w_chk, _ = gd_linear(A_tr, y_tr, lam=0.05, n_iter=8000)
print("GD (λ=0,05) ≈ fechado (λ=0,05)?", np.allclose(w_chk, ridge_fechado(A_tr, y_tr, 0.05), atol=1e-3))
print(f"λ ótimo (CV 5-fold) = {lam_otimo:.4f}  |  R² no teste (Ridge) = {r2_ridge:.3f}")

plt.figure(figsize=(7, 4))
for k, c in enumerate(cols):
    plt.plot(lams, coefs[:, k], label=c.replace("log_", ""), lw=1.2)
plt.axvline(lam_otimo, ls="--", c="gray"); plt.xscale("log")
plt.xlabel("λ"); plt.ylabel("coeficiente"); plt.title("Ridge: coeficientes × λ"); plt.legend(fontsize=6, ncol=2)
plt.tight_layout(); plt.savefig("figuras/etapa2_ridge_path.png", dpi=150)

GD (λ=0,05) ≈ fechado (λ=0,05)? True
λ ótimo (CV 5-fold) = 0.0838  |  R² no teste (Ridge) = 0.762


## 4. Regressão logística (sigmoide + gradiente, do zero) para presença de facção

In [6]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

def gd_logistica(A, y, alpha=0.3, n_iter=5000, balancear=True):
    w = np.zeros(A.shape[1])
    pesos = np.where(y == 1, len(y) / (2 * y.sum()), len(y) / (2 * (len(y) - y.sum()))) if balancear else np.ones(len(y))
    for _ in range(n_iter):
        w += alpha * A.T @ (pesos * (y - sigmoide(A @ w))) / len(y)
    return w

w_log = gd_logistica(A_tr, yb_tr)
pred_log = (sigmoide(A_te @ w_log) >= 0.5).astype(int)
acc_log, bal_log = accuracy_score(yb_te, pred_log), balanced_accuracy_score(yb_te, pred_log)
print(f"Logística própria: acurácia={acc_log:.3f}  balanceada={bal_log:.3f}  F1={f1_score(yb_te, pred_log):.3f}")

Logística própria: acurácia=0.759  balanceada=0.737  F1=0.642


## 5. Comparação com o scikit-learn

In [7]:
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
lr = LinearRegression().fit(Z_tr, y_tr)
rg = Ridge(alpha=lam_otimo * len(Z_tr)).fit(Z_tr, y_tr)
lg = LogisticRegression(class_weight="balanced", max_iter=5000, C=1e6).fit(Z_tr, yb_tr)
p_lg = lg.predict(Z_te)
lr_uni = LinearRegression().fit(x_tr.reshape(-1, 1), y_tr)
pd.DataFrame({
    "Modelo": ["Regressão linear simples", "Regressão multivariada (GD)", "Ridge L2 (λ ótimo)", "Regressão logística"],
    "R² (próprio)": [r2_uni, r2_multi, r2_ridge, np.nan],
    "R² (sklearn)": [lr_uni.score(x_te.reshape(-1, 1), y_te), lr.score(Z_te, y_te), rg.score(Z_te, y_te), np.nan],
    "Acurácia (próprio)": [np.nan] * 3 + [acc_log],
    "Acurácia (sklearn)": [np.nan] * 3 + [accuracy_score(yb_te, p_lg)],
    "Acur. bal. (próprio)": [np.nan] * 3 + [bal_log],
    "Acur. bal. (sklearn)": [np.nan] * 3 + [balanced_accuracy_score(yb_te, p_lg)],
}).round(3)

,Modelo,R² (próprio),R² (sklearn),Acurácia (próprio),Acurácia (sklearn),Acur. bal. (próprio),Acur. bal. (sklearn)
0,Regressão linear simples,0.672,0.672,NaN,NaN,NaN,NaN
1,Regressão multivariada (GD),0.763,0.763,NaN,NaN,NaN,NaN
2,Ridge L2 (λ ótimo),0.762,0.762,NaN,NaN,NaN,NaN
3,Regressão logística,NaN,NaN,0.759,0.759,0.737,0.737
